# שיעור 12 - צמצום היסטוריית הצ'אט עם פנקס סוכנת

פנקס זה מדגים כיצד לנהל הקשר בשיחות ארוכות באמצעות Microsoft Agent Framework. ככל שהשיחות מתארכות, מספר הטוקנים גדל — ובסופו של דבר חורג מחלון ההקשר של המודל. אנו מתמודדים עם זאת בעזרת **תבנית סיכום הקשר** ו-**פנקס סוכנת** לזיכרון מתמשך.

## מה תלמדו:
1. **מדוע ניהול הקשר חשוב**: הבנת מגבלות הטוקנים וחלונות ההקשר
2. **סוכנים מודעי הקשר**: בניית סוכנים שמנהלים את הקשר השיחה שלהם עצמם
3. **תבנית סיכום הקשר**: שימוש בכלים לצמצום היסטוריית השיחה
4. **פנקס סוכנת**: זיכרון מתמשך ששרוד צמצום ההקשר

## דרישות מוקדמות:
- התקנת Azure OpenAI עם משתני סביבה מוגדרים
- הבנה של מושגי סוכנים בסיסיים מהשיעורים הקודמים


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## מדוע ניהול הקשר חשוב

לכל LLM יש **חלון הקשר** סופי — מספר מקסימלי של טוקנים שהוא יכול לעבד בבקשה אחת. ככל ששיחה מרובת סיבובים מתקדמת:

- **מספר הטוקנים גדל בקו ישר** עם כל הודעת משתמש ותשובת עוזר.
- **טוקני ההנחיה שורצים את רוב העלות** כי כל ההיסטוריה נשלחת מחדש בכל סיבוב.
- בסופו של דבר השיחה **עוברת את חלון ההקשר** והמודל או מקצר או נותן שגיאה.

### אסטרטגיות לניהול ההקשר

| אסטרטגיה | איך זה עובד | פשרה |
|---|---|---|
| **קיצוץ** | מחיקת הודעות ישנות | איבוד הקשר מוקדם |
| **סיכום** | דחיסת הודעות ישנות לסיכום | חלק מהפרטים אובדים, אבל נקודות מפתח נשמרות |
| **פד רשימות / זיכרון חיצוני** | אחסון עובדות מפתח מחוץ לשיחה | דורש קריאות לכלים, אך שורד כל קיצור |

בדף זה אנו משלבים **סיכום** עם כלי **פד רשימות** כדי שהסוכן יוכל לשמור על רצף גם כאשר היסטוריית השיחה מדוכסת.


## יצירת סוכן מודע להקשר


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## סימולציה של שיחה ארוכה

בואו נעבור דרך שיחה מרובת סיבובים כדי לראות כיצד ההקשר מצטבר. הסוכן צריך לשמור על פרטים מרכזיים (העדפות, תקציב, תאריכי נסיעה) לאורך הסיבובים ולהדגים רצף.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

שים לב כיצד הסוכן שומר על הקשר מהפניות קודמות — הוא יודע על יפן, סושי, מקדשים, צילום, תקציב של 3000 דולר, טיול לבד, ותקופת אפריל. בשיחה קצרה זה עובד טוב, אך ככל שהשיחה מתארכת, ההיסטוריה המלאה הופכת ליקרה לשליחה מחדש.

בוא נמשיך את השיחה עם פניות נוספות כדי לראות את הצטברות ההקשר:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## תבנית סיכום הקשר

ככל שהשיחה מתארכת, ניתן להשתמש ב**כלי סיכום** כדי לדחוס את ההקשר שנצבר לפורמט תמציתי. הסוכן קורא לכלי זה כדי לתעד העדפות מרכזיות כך שגם אם הודעות ישנות יותר יוסרו, המידע העיקרי יישמר.

תבנית זו היא אבני הבניין לצמצום היסטוריה מתוחכם יותר:
1. הסוכן מזהה עובדות מרכזיות מהשיחה
2. הוא קורא לכלי הסיכום כדי לשמר אותן
3. ניתן להסיר הודעות ישנות בבטחה כי הסיכום תופס את מה שחשוב

להלן הגדרנו כלי `summarize_preferences` שהסוכן יכול לקרוא לו כדי לתעד סיכום תמציתי של מה שלמד.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## סיכום

בשיעור זה למדת כיצד לנהל הקשר בשיחות סוכנים ארוכות טווח באמצעות Microsoft Agent Framework:

### מושגים מרכזיים
- **חלונות הקשר הם סופיים** — כל טוקן בהיסטוריית השיחה עולה כסף ונחשב לגבול.
- **כלי סיכום** מאפשרים לסוכן לדחוס הקשר מצטבר לתמציות קומפקטיות, ומאפשרים הפחתת שימוש בטוקנים תוך שמירה על המידע החשוב.
- **מחברות סוכן** מספקות זיכרון חיצוני מתמשך שמשרוד כל קיצור שיחה.

### מה שבנית
- **סוכן מודע להקשר** שמוודא רצף בשיחות מרובות סיבובים
- **כלי סיכום** (`summarize_preferences`) שרושם פרטים מרכזיים של המשתמש בפורמט מקוצר
- **שיחת מרובה סיבובים** המדגימה שמירה על הקשר וטיפול בשינויים

### יישומים בעולם האמיתי
- **בוטים לשירות לקוחות**: זוכרים העדפות לאורך מפגשי תמיכה ארוכים
- **עוזרים אישיים**: עוקבים אחרי פרויקטים מתמשכים בלי צורך להסביר מחדש את ההקשר
- **מורים פרטיים**: שומרים על התקדמות התלמידים לאורך אינטראקציות רבות

### הצעדים הבאים
- ליישם כלי מחברת מלא עם שטח אחסון מבוסס קבצים
- להוסיף קיצור היסטוריה אוטומטי לאחר סיכום
- לשלב עם מסדי נתונים וקטוריים לחיפוש זיכרון סמנטי
- לבנות סוכנים שיכולים להמשיך שיחות ימים לאחר מכן עם כל ההקשר


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**הצהרת הגבלה**:  
מסמך זה תורגם באמצעות שירות תרגום בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש להיות מודעים לכך שתיקונים אוטומטיים עשויים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפתו המקורית צריך להיחשב כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי אנושי. אנחנו לא אחראים לכל אי-הבנה או פרשנות שגויה הנובעים משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
